In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

In [3]:
# 1. Load the raw data into memory
df = pd.read_csv("shop_smart_ecommerce.csv")


In [4]:
# 2. Prevent Feature Leakage (Drop the 'answer key')
# We do this immediately so it never accidentally touches our model
df = df.drop(columns=["PageValues"])

In [5]:

# 3. Separate Features (X) and Target (y)
X = df.drop(columns=["Revenue"])
y = df["Revenue"].astype(int)  # Convert boolean to int (True=1, False=0)


In [6]:
# 4. The Engineering Split (80% Training, 20% Testing)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Training Data Shape: {X_train.shape}")
print(f"Testing Data Shape: {X_test.shape}")


Training Data Shape: (9864, 16)
Testing Data Shape: (2466, 16)


In [7]:
# Dynamically extract column names based on their data type
# We exclude the target variable 'Revenue' because it's not a feature
numeric_features = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_featues = X_train.select_dtypes(include=["object", "bool"]).columns.tolist()

print(f"Detected {len(numeric_features)} numeric features: {numeric_features}")
print(f"Detected {len(categorical_featues)} categorical features: {categorical_featues}")

Detected 13 numeric features: ['Administrative', 'Administrative_Duration', 'Informational', 'Informational_Duration', 'ProductRelated', 'ProductRelated_Duration', 'BounceRates', 'ExitRates', 'SpecialDay', 'OperatingSystems', 'Browser', 'Region', 'TrafficType']
Detected 3 categorical features: ['Month', 'VisitorType', 'Weekend']


In [8]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.tree import DecisionTreeClassifier


# 1. Define the Transformers (The Pre-cooks)
numeric_transformer = StandardScaler()
categorical_transformer = OneHotEncoder(handle_unknown="ignore")

# 2. Bundle them into the Preprocessor (The Kitchen Manager)
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_featues),
    ]
)

# 3. Assemble the Master Pipeline (The Full Factory)
# we added class_weight='balanced' to fix the imbalance

pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", DecisionTreeClassifier(class_weight="balanced", random_state=42)),
    ]
)

print("Pipeline successfully created. Here's the structure: \n", pipeline)

Pipeline successfully created. Here's the structure: 
 Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['Administrative',
                                                   'Administrative_Duration',
                                                   'Informational',
                                                   'Informational_Duration',
                                                   'ProductRelated',
                                                   'ProductRelated_Duration',
                                                   'BounceRates', 'ExitRates',
                                                   'SpecialDay',
                                                   'OperatingSystems',
                                                   'Browser', 'Region',
                                                   'TrafficType']),
                                           

In [10]:
from sklearn.model_selection import GridSearchCV

# 1. Define the Grid of options to test
# IMPORTANT: Because our Decision Tree is inside a Pipeline named 'classifier',
# we MUST put 'classifier__' in front of the parameter names so the GridSearch
# knows exactly which part of the pipeline to change.

param_grid = {
    "classifier__max_depth": [4, 6, 8, 10],
    "classifier__min_samples_leaf": [20, 30, 50, 100]
}

# 2. Build the Optimization Engine
# scoring="f1" tells it to optimize for the exact metric the business needs!

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring="f1",
    cv=5, # Cross-validation: tests each combination 5 different times to be sure
    n_jobs=-1 # Use all cores to run it incrdibly fast and speed up search
)

# 3. Train the Engine (This is where the heavy lifting happens)
print("Training models and searching for the best parameters. Please wait...")
grid_search.fit(X_train, y_train)

# 4. Extract the Winner
best_model = grid_search.best_estimator_
print(f"Best Parameters Found: {grid_search.best_params_}")
print(f"Best Training F1 Score: {grid_search.best_score_:.4f}")

Training models and searching for the best parameters. Please wait...
Best Parameters Found: {'classifier__max_depth': 4, 'classifier__min_samples_leaf': 20}
Best Training F1 Score: 0.3977


In [11]:
from sklearn.metrics import classification_report, confusion_matrix

# 1. The Inference Step (Predicting on unseen data)
y_pred = best_model.predict(X_test)

# 2. The Engineering Report
print("--- PRODUCTION EVALUATION REPORT ---")
# This prints Precision, Recall, and F1 for BOTH classes (0 = No Buy, 1 = Buy)
print(classification_report(y_test, y_pred))

# 3. The Confusion Matrix (Where did we make our mistakes?)
print("\n--- CONFUSION MATRIX ---")
print(confusion_matrix(y_test, y_pred))

--- PRODUCTION EVALUATION REPORT ---
              precision    recall  f1-score   support

           0       0.91      0.74      0.82      2084
           1       0.30      0.61      0.40       382

    accuracy                           0.72      2466
   macro avg       0.61      0.68      0.61      2466
weighted avg       0.82      0.72      0.75      2466


--- CONFUSION MATRIX ---
[[1543  541]
 [ 149  233]]
